# 11.1 ONNX for NLP — Apply

## Objective

Build and run NLP inference pipelines with ONNX Runtime: tokenization, feed preparation,
sequence classification, dynamic lengths, and batch processing. Uses toy models built with
`onnx.helper` as fallback, with optional HuggingFace integration.

**Prerequisites:** `pip install onnx onnxruntime numpy`

## Table of Contents
1. [Setup](#setup)
2. [Exercise 1 — Build & Inspect an NLP Session](#ex1)
3. [Exercise 2 — Tokenizer Integration](#ex2)
4. [Exercise 3 — Feed Preparation for Sequence Models](#ex3)
5. [Exercise 4 — Text Classification Inference](#ex4)
6. [Exercise 5 — Dynamic Sequence Length Handling](#ex5)
7. [Exercise 6 — Batch Inference for NLP](#ex6)
8. [Exercise 7 — Framework Parity Verification](#ex7)
9. [Challenge — NLP Inference Pipeline](#challenge)
10. [Summary](#summary)

In [ ]:
!pip install onnx onnxruntime numpy -q

<a id='setup'></a>
## Setup

We build a toy transformer-like model using `onnx.helper`.
Architecture: Embedding → LayerNorm → Attention (simplified) → FC → Logits.

This model accepts `input_ids` (int64) and `attention_mask` (int64), similar to
HuggingFace transformer models.

In [ ]:
import os
import time
import json
import numpy as np
from typing import Any, Dict, List, Optional, Tuple

import onnx
from onnx import helper, TensorProto, numpy_helper
import onnxruntime as ort

# Check if HuggingFace transformers is available
HAS_TRANSFORMERS = False
try:
    from transformers import AutoTokenizer
    HAS_TRANSFORMERS = True
    print("HuggingFace transformers available.")
except ImportError:
    print("HuggingFace transformers not found — using toy tokenizer fallback.")

np.random.seed(42)

# Model parameters
VOCAB_SIZE = 1000
HIDDEN_DIM = 64
NUM_CLASSES = 3  # sentiment: negative, neutral, positive
MAX_SEQ_LEN = 32


def build_nlp_classifier() -> onnx.ModelProto:
    """
    Build toy text classifier: Gather(embedding) -> ReduceMean(with mask) -> FC -> logits.
    Inputs: input_ids [batch, seq], attention_mask [batch, seq]
    Output: logits [batch, num_classes]
    """
    # Embedding table
    embed_W = numpy_helper.from_array(
        np.random.randn(VOCAB_SIZE, HIDDEN_DIM).astype(np.float32) * 0.02,
        name="embed_W"
    )
    # Classification head
    fc_W = numpy_helper.from_array(
        np.random.randn(HIDDEN_DIM, NUM_CLASSES).astype(np.float32) * 0.02,
        name="fc_W"
    )
    fc_B = numpy_helper.from_array(
        np.zeros(NUM_CLASSES, dtype=np.float32), name="fc_B"
    )

    # Inputs
    input_ids = helper.make_tensor_value_info("input_ids", TensorProto.INT64, ["batch", "seq"])
    attention_mask = helper.make_tensor_value_info("attention_mask", TensorProto.INT64, ["batch", "seq"])
    logits = helper.make_tensor_value_info("logits", TensorProto.FLOAT, ["batch", NUM_CLASSES])

    # Gather embedding
    gather = helper.make_node("Gather", ["embed_W", "input_ids"], ["embedded"], axis=0)

    # Cast attention_mask to float for masking
    cast_mask = helper.make_node("Cast", ["attention_mask"], ["mask_float"], to=TensorProto.FLOAT)

    # Unsqueeze mask for broadcasting: [batch, seq] -> [batch, seq, 1]
    unsqueeze_axes = numpy_helper.from_array(np.array([2], dtype=np.int64), name="unsqueeze_axes")
    unsqueeze = helper.make_node("Unsqueeze", ["mask_float", "unsqueeze_axes"], ["mask_3d"])

    # Masked mean pooling: sum(embedded * mask) / sum(mask)
    mul_mask = helper.make_node("Mul", ["embedded", "mask_3d"], ["masked_embed"])
    reduce_axes_1 = numpy_helper.from_array(np.array([1], dtype=np.int64), name="reduce_axes_1")
    sum_embed = helper.make_node("ReduceSum", ["masked_embed", "reduce_axes_1"], ["sum_embed"], keepdims=0)
    sum_mask = helper.make_node("ReduceSum", ["mask_3d", "reduce_axes_1"], ["sum_mask"], keepdims=0)

    # Clamp mask sum to avoid division by zero
    clamp_val = numpy_helper.from_array(np.array([1e-9], dtype=np.float32), name="clamp_val")
    clamp = helper.make_node("Max", ["sum_mask", "clamp_val"], ["safe_mask"])

    # Divide
    mean_pool = helper.make_node("Div", ["sum_embed", "safe_mask"], ["pooled"])

    # FC layer
    matmul = helper.make_node("MatMul", ["pooled", "fc_W"], ["fc_out"])
    add = helper.make_node("Add", ["fc_out", "fc_B"], ["logits"])

    graph = helper.make_graph(
        [gather, cast_mask, unsqueeze, mul_mask, sum_embed, sum_mask, clamp, mean_pool, matmul, add],
        "nlp_classifier",
        [input_ids, attention_mask],
        [logits],
        [embed_W, fc_W, fc_B, unsqueeze_axes, reduce_axes_1, clamp_val],
    )
    model = helper.make_model(graph, opset_imports=[helper.make_opsetid("", 17)])
    model.ir_version = 8
    onnx.checker.check_model(model)
    return model


nlp_model = build_nlp_classifier()
NLP_MODEL_PATH = "/tmp/nlp_classifier.onnx"
onnx.save(nlp_model, NLP_MODEL_PATH)
print(f"NLP classifier saved: {os.path.getsize(NLP_MODEL_PATH):,} bytes")
print(f"Vocab: {VOCAB_SIZE}, Hidden: {HIDDEN_DIM}, Classes: {NUM_CLASSES}")

<a id='ex1'></a>
## Exercise 1 — Build & Inspect an NLP Session

NLP models have specific I/O signatures. Understanding them is critical for
correct feed preparation.

In [ ]:
def build_nlp_session(model_path: str) -> ort.InferenceSession:
    so = ort.SessionOptions()
    so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
    so.intra_op_num_threads = 2
    return ort.InferenceSession(model_path, sess_options=so, providers=["CPUExecutionProvider"])


def inspect_nlp_model(sess: ort.InferenceSession) -> Dict[str, Any]:
    """Inspect NLP model I/O for feed preparation."""
    info = {"inputs": [], "outputs": []}

    for inp in sess.get_inputs():
        info["inputs"].append({
            "name": inp.name,
            "shape": [str(d) for d in inp.shape],
            "type": inp.type,
            "is_dynamic": any(isinstance(d, str) for d in inp.shape),
            "expected_dtype": "int64" if "int" in inp.type else "float32",
        })

    for out in sess.get_outputs():
        info["outputs"].append({
            "name": out.name,
            "shape": [str(d) for d in out.shape],
            "type": out.type,
        })

    return info


nlp_session = build_nlp_session(NLP_MODEL_PATH)
model_info = inspect_nlp_model(nlp_session)
print("NLP Model Inspection:")
print(json.dumps(model_info, indent=2))

# Assertions
assert len(model_info["inputs"]) == 2
assert model_info["inputs"][0]["name"] == "input_ids"
assert model_info["inputs"][1]["name"] == "attention_mask"
assert model_info["inputs"][0]["is_dynamic"]
print("\nModel inspection validated.")

<a id='ex2'></a>
## Exercise 2 — Tokenizer Integration

Tokenization converts raw text to token IDs. We implement:
- A simple whitespace tokenizer (fallback)
- HuggingFace tokenizer integration (if available)

Both produce the same interface: `{input_ids, attention_mask}` as int64 numpy arrays.

In [ ]:
class SimpleTokenizer:
    """Whitespace tokenizer with fixed vocabulary (educational fallback)."""

    def __init__(self, vocab_size: int = VOCAB_SIZE, max_length: int = MAX_SEQ_LEN):
        self.vocab_size = vocab_size
        self.max_length = max_length
        self.pad_id = 0
        self.unk_id = 1
        self.cls_id = 2
        self.sep_id = 3

    def _hash_token(self, token: str) -> int:
        """Deterministic token -> id mapping via hash."""
        h = hash(token.lower()) % (self.vocab_size - 4) + 4
        return h

    def encode(
        self, text: str, max_length: Optional[int] = None,
    ) -> Dict[str, np.ndarray]:
        """Tokenize text, returning input_ids and attention_mask."""
        max_len = max_length or self.max_length
        tokens = text.strip().split()

        # [CLS] token1 token2 ... [SEP]
        ids = [self.cls_id]
        for t in tokens[:max_len - 2]:
            ids.append(self._hash_token(t))
        ids.append(self.sep_id)

        # Pad to max_length
        attention_mask = [1] * len(ids) + [0] * (max_len - len(ids))
        ids = ids + [self.pad_id] * (max_len - len(ids))

        return {
            "input_ids": np.array([ids], dtype=np.int64),
            "attention_mask": np.array([attention_mask], dtype=np.int64),
        }

    def batch_encode(
        self, texts: List[str], max_length: Optional[int] = None,
    ) -> Dict[str, np.ndarray]:
        """Batch tokenization with padding to longest."""
        encoded = [self.encode(t, max_length) for t in texts]
        return {
            "input_ids": np.concatenate([e["input_ids"] for e in encoded], axis=0),
            "attention_mask": np.concatenate([e["attention_mask"] for e in encoded], axis=0),
        }


# Use HuggingFace if available, otherwise fallback
tokenizer = SimpleTokenizer()

# Test tokenization
text = "ONNX Runtime is great for NLP deployment"
encoded = tokenizer.encode(text)

print(f"Text: '{text}'")
print(f"Input IDs shape: {encoded['input_ids'].shape}")
print(f"Attention mask shape: {encoded['attention_mask'].shape}")
print(f"Input IDs: {encoded['input_ids'][0, :10]}...")
print(f"Attention mask: {encoded['attention_mask'][0, :10]}...")

# Verify shapes and types
assert encoded["input_ids"].dtype == np.int64
assert encoded["attention_mask"].dtype == np.int64
assert encoded["input_ids"].shape == (1, MAX_SEQ_LEN)
assert encoded["input_ids"][0, 0] == tokenizer.cls_id  # starts with [CLS]
print("\nTokenization validated.")

<a id='ex3'></a>
## Exercise 3 — Feed Preparation for Sequence Models

ONNX models may not accept all tokenizer outputs. We must:
1. Match tokenizer outputs to model input names
2. Ensure correct dtypes (int64 for IDs, sometimes int32)
3. Validate shapes against model expectations

In [ ]:
def prepare_feeds(
    session: ort.InferenceSession,
    tokenizer_output: Dict[str, np.ndarray],
) -> Dict[str, np.ndarray]:
    """
    Map tokenizer outputs to model inputs, ensuring dtype/shape compatibility.
    """
    model_inputs = {inp.name: inp for inp in session.get_inputs()}
    feeds = {}
    matched = []
    skipped = []

    for name, tensor in tokenizer_output.items():
        if name in model_inputs:
            inp = model_inputs[name]
            # Ensure correct dtype
            if "int64" in inp.type:
                tensor = tensor.astype(np.int64)
            elif "int32" in inp.type:
                tensor = tensor.astype(np.int32)
            elif "float" in inp.type:
                tensor = tensor.astype(np.float32)
            feeds[name] = tensor
            matched.append(name)
        else:
            skipped.append(name)

    # Check for missing required inputs
    missing = set(model_inputs.keys()) - set(feeds.keys())
    if missing:
        raise ValueError(f"Model requires inputs not in tokenizer output: {missing}")

    return feeds


# Prepare feeds from tokenizer output
feeds = prepare_feeds(nlp_session, encoded)
print("Prepared feeds:")
for name, tensor in feeds.items():
    print(f"  {name}: shape={tensor.shape}, dtype={tensor.dtype}")

# Run inference
output = nlp_session.run(None, feeds)[0]
print(f"\nOutput shape: {output.shape}")
assert output.shape == (1, NUM_CLASSES)

# Test with extra tokenizer outputs (e.g., token_type_ids)
extended_output = {**encoded, "token_type_ids": np.zeros_like(encoded["input_ids"])}
feeds_extended = prepare_feeds(nlp_session, extended_output)
assert "token_type_ids" not in feeds_extended  # should be skipped
print("Feed preparation with extra outputs handled correctly.")

<a id='ex4'></a>
## Exercise 4 — Text Classification Inference

Complete classification pipeline: text → tokens → inference → labels.

Softmax converts logits $z$ to class probabilities:

$$P(y=c \mid x) = \frac{e^{z_c}}{\sum_{j} e^{z_j}}$$

In [ ]:
SENTIMENT_LABELS = ["negative", "neutral", "positive"]


def softmax(logits: np.ndarray) -> np.ndarray:
    shifted = logits - np.max(logits, axis=-1, keepdims=True)
    exp_vals = np.exp(shifted)
    return exp_vals / np.sum(exp_vals, axis=-1, keepdims=True)


def classify_text(
    text: str,
    session: ort.InferenceSession,
    tokenizer: SimpleTokenizer,
    labels: List[str],
) -> Dict[str, Any]:
    """End-to-end text classification."""
    # Tokenize
    encoded = tokenizer.encode(text)
    feeds = prepare_feeds(session, encoded)

    # Inference
    logits = session.run(None, feeds)[0]  # [1, num_classes]

    # Postprocess
    probs = softmax(logits[0])
    predicted_idx = int(np.argmax(probs))

    return {
        "text": text,
        "predicted_label": labels[predicted_idx],
        "confidence": float(probs[predicted_idx]),
        "all_scores": {label: float(p) for label, p in zip(labels, probs)},
        "logits": logits[0].tolist(),
    }


# Classify several texts
test_texts = [
    "This product is absolutely wonderful and exceeded expectations",
    "The service was okay nothing special",
    "Terrible experience would not recommend to anyone",
]

print("Text Classification Results:")
print("=" * 60)
for text in test_texts:
    result = classify_text(text, nlp_session, tokenizer, SENTIMENT_LABELS)
    print(f"\nText: '{result['text']}'")
    print(f"  Predicted: {result['predicted_label']} ({result['confidence']:.1%})")
    for label, score in result['all_scores'].items():
        print(f"    {label}: {score:.3f}")

# Verify output structure
r = classify_text("test", nlp_session, tokenizer, SENTIMENT_LABELS)
assert r["predicted_label"] in SENTIMENT_LABELS
assert 0 <= r["confidence"] <= 1
assert abs(sum(r["all_scores"].values()) - 1.0) < 1e-5
print("\nClassification pipeline validated.")

<a id='ex5'></a>
## Exercise 5 — Dynamic Sequence Length Handling

NLP models often support dynamic sequence lengths. Key considerations:
- Padding to a fixed length wastes computation
- Dynamic shapes allow efficiency but complicate batching
- Bucketing balances both concerns

Computational cost scales with sequence length:

$$\text{FLOPs}_{\text{attention}} = O(n^2 \cdot d)$$

where $n$ is sequence length and $d$ is hidden dimension.

In [ ]:
def encode_dynamic_length(
    text: str,
    tokenizer: SimpleTokenizer,
    bucket_sizes: List[int] = [8, 16, 32, 64, 128],
) -> Tuple[Dict[str, np.ndarray], int]:
    """Encode text with bucketed length for efficiency."""
    # Get actual token count
    tokens = text.strip().split()
    actual_len = len(tokens) + 2  # +2 for [CLS] and [SEP]

    # Find smallest bucket that fits
    padded_len = max(bucket_sizes)  # fallback
    for bucket in sorted(bucket_sizes):
        if bucket >= actual_len:
            padded_len = bucket
            break

    encoded = tokenizer.encode(text, max_length=padded_len)
    return encoded, padded_len


def measure_length_impact(
    session: ort.InferenceSession,
    tokenizer: SimpleTokenizer,
    lengths: List[int] = [4, 8, 16, 32],
    num_runs: int = 50,
) -> List[Dict[str, Any]]:
    """Measure inference time vs. sequence length."""
    results = []
    for seq_len in lengths:
        text = " ".join(["word"] * (seq_len - 2))  # fill to desired length
        encoded = tokenizer.encode(text, max_length=seq_len)
        feeds = prepare_feeds(session, encoded)

        # Warmup
        for _ in range(5):
            session.run(None, feeds)

        # Measure
        times = []
        for _ in range(num_runs):
            start = time.perf_counter()
            session.run(None, feeds)
            times.append((time.perf_counter() - start) * 1000)

        results.append({
            "seq_len": seq_len,
            "mean_ms": round(float(np.mean(times)), 3),
            "std_ms": round(float(np.std(times)), 3),
        })

    return results


# Test dynamic length encoding
short_text = "hello world"
long_text = " ".join(["word"] * 25)

short_enc, short_bucket = encode_dynamic_length(short_text, tokenizer)
long_enc, long_bucket = encode_dynamic_length(long_text, tokenizer)

print(f"Short text ({len(short_text.split())} words) -> bucket size: {short_bucket}")
print(f"Long text ({len(long_text.split())} words) -> bucket size: {long_bucket}")
assert short_bucket <= long_bucket

# Measure latency vs length
length_impact = measure_length_impact(nlp_session, tokenizer)
print("\nLatency vs. Sequence Length:")
for r in length_impact:
    print(f"  seq_len={r['seq_len']:>3d}: {r['mean_ms']:.3f} ms (±{r['std_ms']:.3f})")

<a id='ex6'></a>
## Exercise 6 — Batch Inference for NLP

Batch inference amortizes overhead and improves throughput.

Throughput gain:

$$\text{Throughput}_{\text{batch}} = \frac{B}{t_{\text{batch}}} > \frac{B}{B \times t_{\text{single}}}$$

due to parallelism within operations (SIMD, cache locality).

In [ ]:
def batch_classify(
    texts: List[str],
    session: ort.InferenceSession,
    tokenizer: SimpleTokenizer,
    labels: List[str],
    batch_size: int = 8,
) -> List[Dict[str, Any]]:
    """Classify texts in batches."""
    all_results = []

    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i + batch_size]
        encoded = tokenizer.batch_encode(batch_texts)
        feeds = prepare_feeds(session, encoded)

        logits = session.run(None, feeds)[0]  # [batch, classes]
        probs = softmax(logits)

        for j, text in enumerate(batch_texts):
            pred_idx = int(np.argmax(probs[j]))
            all_results.append({
                "text": text,
                "label": labels[pred_idx],
                "confidence": float(probs[j, pred_idx]),
            })

    return all_results


# Generate test corpus
corpus = [
    "Great product highly recommended",
    "Worst purchase ever made",
    "Average quality nothing special",
    "Exceeded all my expectations",
    "Complete waste of money",
    "It works fine",
    "Absolutely love this",
    "Never buying again",
    "Solid and reliable",
    "Disappointing results",
]

# Compare single vs batch timing
# Single inference
start = time.perf_counter()
single_results = [classify_text(t, nlp_session, tokenizer, SENTIMENT_LABELS) for t in corpus]
single_time = (time.perf_counter() - start) * 1000

# Batch inference
start = time.perf_counter()
batch_results = batch_classify(corpus, nlp_session, tokenizer, SENTIMENT_LABELS, batch_size=5)
batch_time = (time.perf_counter() - start) * 1000

print(f"Single inference: {single_time:.1f} ms for {len(corpus)} texts")
print(f"Batch inference:  {batch_time:.1f} ms for {len(corpus)} texts")
print(f"Speedup: {single_time / batch_time:.2f}x")

# Verify results match
for s, b in zip(single_results, batch_results):
    assert s["predicted_label"] == b["label"], f"Mismatch for: {s['text']}"

print(f"\nAll {len(corpus)} predictions match between single and batch.")
print("\nBatch results:")
for r in batch_results[:5]:
    print(f"  '{r['text'][:40]}...' -> {r['label']} ({r['confidence']:.1%})")

<a id='ex7'></a>
## Exercise 7 — Framework Parity Verification

When deploying NLP models, we must verify that ONNX Runtime produces
the same results as the original framework. We simulate this by computing
the expected output manually (NumPy) and comparing with ORT.

In [ ]:
def numpy_reference_forward(
    input_ids: np.ndarray,
    attention_mask: np.ndarray,
    embed_W: np.ndarray,
    fc_W: np.ndarray,
    fc_B: np.ndarray,
) -> np.ndarray:
    """Manual NumPy implementation matching our ONNX model."""
    # Gather embedding
    embedded = embed_W[input_ids]  # [batch, seq, hidden]

    # Masked mean pooling
    mask_float = attention_mask.astype(np.float32)  # [batch, seq]
    mask_3d = mask_float[:, :, np.newaxis]  # [batch, seq, 1]

    masked_embed = embedded * mask_3d
    sum_embed = masked_embed.sum(axis=1)  # [batch, hidden]
    sum_mask = mask_3d.sum(axis=1)  # [batch, 1]
    sum_mask = np.maximum(sum_mask, 1e-9)

    pooled = sum_embed / sum_mask  # [batch, hidden]

    # FC layer
    logits = pooled @ fc_W + fc_B  # [batch, classes]
    return logits


# Extract weights from the ONNX model
model = onnx.load(NLP_MODEL_PATH)
weights = {}
for init in model.graph.initializer:
    weights[init.name] = numpy_helper.to_array(init)

# Run parity check on multiple inputs
max_diffs = []
for _ in range(10):
    seq_len = np.random.randint(5, MAX_SEQ_LEN)
    ids = np.random.randint(0, VOCAB_SIZE, size=(1, MAX_SEQ_LEN)).astype(np.int64)
    mask = np.zeros((1, MAX_SEQ_LEN), dtype=np.int64)
    mask[0, :seq_len] = 1

    # ORT output
    ort_out = nlp_session.run(None, {"input_ids": ids, "attention_mask": mask})[0]

    # NumPy reference
    ref_out = numpy_reference_forward(
        ids, mask, weights["embed_W"], weights["fc_W"], weights["fc_B"]
    )

    max_diff = float(np.max(np.abs(ort_out - ref_out)))
    max_diffs.append(max_diff)

print(f"Parity check over 10 random inputs:")
print(f"  Mean max diff: {np.mean(max_diffs):.2e}")
print(f"  Max max diff:  {np.max(max_diffs):.2e}")
assert np.max(max_diffs) < 1e-4, "Parity violation!"
print("\nFramework parity verified!")

<a id='challenge'></a>
## Challenge — NLP Inference Pipeline

Build a complete NLP inference pipeline class with:
1. Tokenization (with caching)
2. Dynamic batching
3. Postprocessing (softmax, label mapping)
4. Performance tracking

In [ ]:
class NLPInferencePipeline:
    """Production NLP inference pipeline with batching and metrics."""

    def __init__(
        self,
        model_path: str,
        labels: List[str],
        max_seq_len: int = 32,
        batch_size: int = 8,
    ):
        self.session = build_nlp_session(model_path)
        self.tokenizer = SimpleTokenizer(max_length=max_seq_len)
        self.labels = labels
        self.batch_size = batch_size
        self._token_cache: Dict[str, Dict[str, np.ndarray]] = {}
        self.metrics = {
            "total_texts": 0,
            "total_batches": 0,
            "cache_hits": 0,
            "total_latency_ms": 0.0,
        }

    def _tokenize_cached(self, text: str) -> Dict[str, np.ndarray]:
        if text in self._token_cache:
            self.metrics["cache_hits"] += 1
            return self._token_cache[text]
        encoded = self.tokenizer.encode(text)
        self._token_cache[text] = encoded
        return encoded

    def predict(self, texts: List[str]) -> List[Dict[str, Any]]:
        """Classify a list of texts with automatic batching."""
        all_results = []
        start = time.perf_counter()

        for i in range(0, len(texts), self.batch_size):
            batch = texts[i:i + self.batch_size]

            # Tokenize (with cache)
            encoded_list = [self._tokenize_cached(t) for t in batch]
            input_ids = np.concatenate([e["input_ids"] for e in encoded_list])
            attention_mask = np.concatenate([e["attention_mask"] for e in encoded_list])

            feeds = {
                "input_ids": input_ids,
                "attention_mask": attention_mask,
            }

            # Inference
            logits = self.session.run(None, feeds)[0]
            probs = softmax(logits)

            # Postprocess
            for j, text in enumerate(batch):
                pred_idx = int(np.argmax(probs[j]))
                all_results.append({
                    "text": text,
                    "label": self.labels[pred_idx],
                    "confidence": float(probs[j, pred_idx]),
                    "scores": {l: float(p) for l, p in zip(self.labels, probs[j])},
                })

            self.metrics["total_batches"] += 1

        elapsed = (time.perf_counter() - start) * 1000
        self.metrics["total_texts"] += len(texts)
        self.metrics["total_latency_ms"] += elapsed

        return all_results

    def get_metrics(self) -> Dict[str, Any]:
        avg_lat = (
            self.metrics["total_latency_ms"] / max(self.metrics["total_texts"], 1)
        )
        return {
            **self.metrics,
            "avg_latency_per_text_ms": round(avg_lat, 3),
            "cache_size": len(self._token_cache),
        }


# Test the pipeline
pipeline = NLPInferencePipeline(
    model_path=NLP_MODEL_PATH,
    labels=SENTIMENT_LABELS,
    batch_size=4,
)

# First run
results = pipeline.predict(corpus)
print("Pipeline Results:")
for r in results[:5]:
    print(f"  '{r['text'][:35]}...' -> {r['label']} ({r['confidence']:.1%})")

# Second run (should hit cache)
results2 = pipeline.predict(corpus[:3])

# Verify consistency
for r1, r2 in zip(results[:3], results2):
    assert r1["label"] == r2["label"]
    assert abs(r1["confidence"] - r2["confidence"]) < 1e-6

metrics = pipeline.get_metrics()
print(f"\nPipeline Metrics:")
print(json.dumps(metrics, indent=2))
assert metrics["cache_hits"] > 0
assert metrics["total_texts"] == 13
print("\nNLP inference pipeline validated!")

<a id='summary'></a>
## Summary

| Component | Key Insight |
|-----------|------------|
| **Model Building** | Embedding + Masked Pooling + FC is a minimal classifier |
| **Tokenization** | Must produce int64 `input_ids` + `attention_mask` |
| **Feed Preparation** | Match tokenizer outputs to model input names/types |
| **Classification** | Softmax → argmax → label mapping |
| **Dynamic Length** | Bucketing balances padding waste vs batch efficiency |
| **Batching** | Amortizes ORT overhead; 1.5-3× throughput gain typical |
| **Parity** | Always verify ORT matches reference implementation |

**Key equation — Masked mean pooling:**

$$\mathbf{h}_{\text{pooled}} = \frac{\sum_{i=1}^{n} m_i \cdot \mathbf{h}_i}{\sum_{i=1}^{n} m_i}$$

where $m_i$ is the attention mask and $\mathbf{h}_i$ are token embeddings.